## Create dataset for survival analysis and logistic regression (inc. covariates)
Goal: 
Row = patient
Columns: 

patientuid, 
dob, dob_ordinal, 
date_vax, date_vax_ordinal, 
age_at_vax,
state, 
policy (0 = medical only, 1 = religious exempt, 2 = religious and personal exempt),  party (only Dem, Rep, Bipartisan), 
household_id, 
practiceid, 

state,
county,
rurality, 
neighborhood-level vote share (network effects), 
parent_count (how many potential parents pre-L2 linkage they had)



In [ ]:
import pandas as pd
import zipfile
import ast
import re
import json
import numpy as np
import statistics
import datetime
import matplotlib.pyplot as plt
import gzip
import os
import math

## Load data

In [ ]:
children = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/patients_ruca_svi_2018.csv.zip')

In [ ]:
children.columns

In [ ]:
engage_date = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/intermediate/earliest_engaged.csv.zip')

In [ ]:
engage_date

In [ ]:
child_politics = children[children['patientuid'].isin(engage_date['patientuid'])]

In [ ]:
child_politics.columns

In [ ]:
len(child_politics), len(engage_date)

In [ ]:
len(child_politics)

In [ ]:
child_politics = child_politics[~child_politics['party'].isna()]

In [ ]:
child_politics = child_politics[~child_politics['household_id'].isna()]

Children who engaged with the AFC as a patient in any capacity (beyond simply being listed in the patient register) 

In [ ]:
sum(child_politics['party'] == 'Democratic') + sum(child_politics['party'] == 'Republican')

## Merge in measles data

In [ ]:
vax = pd.read_csv('/share/pi/deho-pi/AFC/mortonc/processed/measles_first_vax_notes_and_code.csv.zip')
vax['vax_date'] = vax['date']
vax = vax[['patientuid', 'vax_date']]

In [ ]:
child_politics = pd.merge(child_politics, vax, how = 'left')

In [ ]:
sum(~child_politics['vax_date'].isna()), len(vax)

In [ ]:
def to_ordinal_with_na(date):
    if pd.isna(date):
        return np.nan  # Return NaN for missing values
    else:
        return date.toordinal()
def myround(x, base=4):
    return base * round(x/base)

In [ ]:
def process_child_politics(child_politics):
    child_politics.loc[:,'dob'] = pd.to_datetime(child_politics['dob'], format='mixed')
    child_politics.loc[:,'dob_ordinal'] = [x.toordinal() for x in child_politics['dob']]

    child_politics.loc[:,'vax_date'] = pd.to_datetime(child_politics['vax_date'], format='mixed')

    child_politics.loc[:,'vax_date_ordinal'] = [to_ordinal_with_na(x) for x in child_politics['vax_date']]

    child_politics.loc[:,'time_to_vax'] = (child_politics['vax_date_ordinal'] - child_politics['dob_ordinal'])/365.0

    # fill in unknown child_politics values with 100, create indicator column with 0s
    child_politics['time_to_vax'] = child_politics['time_to_vax'].fillna(value=100)
    
    return(child_politics)

In [ ]:
child_politics = process_child_politics(child_politics)

In [ ]:
child_politics.columns

In [ ]:
len(child_politics)

## Add parent_count

In [ ]:
# parent_count comes create_families.ipynb
parent_count = np.load('/share/pi/deho-pi/AFC/mortonc/intermediate/parent_count.csv.npy', allow_pickle = True)

In [ ]:
parent_count = pd.DataFrame(parent_count)
parent_count.columns = ['patientuid', 'parent_count']

In [ ]:
child_politics = pd.merge(child_politics, parent_count)

In [ ]:
len(child_politics)

## Add years to engage

In [ ]:
child_politics = pd.merge(child_politics, engage_date, how = 'left')

In [ ]:
len(child_politics)

## Save

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/child_politics_covariates_01202026_2018.csv', child_politics)

In [ ]:
child_politics.columns

In [ ]:
sum(child_politics['svi']==-999)